In [1]:
import numpy as np
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
from dask.distributed import Client
from pystac.extensions.eo import EOExtension as eo
from dask_ml.cluster import SpectralClustering
import pyproj

# Viz
import hvplot.xarray

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

bbox = [-118.89, 38.54, -118.57, 38.84]  # Region over a lake in Nevada, USA
datetime = "2017-06-01/2017-09-30"  # Summer months of 2017
collection = "landsat-c2-l2"
platform = "landsat-8"
cloudy_less_than = 1  # percent

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": cloudy_less_than}, "platform": {"in": [platform]}},
)
items = search.get_all_items()
print(f"Returned {len(items)} Items:")
[[i, item.id] for i, item in enumerate(items)]

c:\Users\jcahi\OneDrive\Desktop\musa-650-spring2026\.venv\Lib\site-packages\pystac_client\item_search.py:940: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


Returned 3 Items:


[[0, 'LC08_L2SP_042033_20170718_02_T1'],
 [1, 'LC08_L2SP_042033_20170702_02_T1'],
 [2, 'LC08_L2SP_042033_20170616_02_T1']]

In [3]:
item = items[1]  # select one of the results

In [4]:
assets = []
for _, asset in item.assets.items():
    try:
        assets.append(asset.extra_fields["eo:bands"][0])
    except:
        pass

cols_ordered = [
    "common_name",
    "description",
    "name",
    "center_wavelength",
    "full_width_half_max",
]
bands = pd.DataFrame.from_dict(assets)[cols_ordered]
bands

,common_name,description,name,center_wavelength,full_width_half_max
0,red,Visible red,OLI_B4,0.65,0.04
1,blue,Visible blue,OLI_B2,0.48,0.06
2,green,Visible green,OLI_B3,0.56,0.06
3,nir08,Near infrared,OLI_B5,0.87,0.03
4,lwir11,Long-wave infrared,TIRS_B10,10.90,0.59
5,swir16,Short-wave infrared,OLI_B6,1.61,0.09
6,swir22,Short-wave infrared,OLI_B7,2.20,0.19
7,coastal,Coastal/Aerosol,OLI_B1,0.44,0.02


In [5]:
ds_2017 = odc.stac.stac_load(
    [item],
    bands=bands.common_name.values,
    bbox=bbox,
    chunks={},  # <-- use Dask
    resolution=100
).isel(time=0)

In [6]:
epsg = item.properties["proj:code"]
ds_2017.attrs["crs"] = f"epsg:{epsg}"

In [7]:
da_2017 = ds_2017.to_array(dim="band")
da_2017

<xarray.DataArray (band: 8, y: 339, x: 285)> Size: 2MB
dask.array<stack, shape=(8, 339, 285), dtype=uint16, chunksize=(1, 339, 285), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
  * y            (y) float64 3kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
  * x            (x) float64 2kB 3.353e+05 3.354e+05 ... 3.636e+05 3.637e+05
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
Attributes:
    crs:      epsg:EPSG:32611

In [8]:
flattened_xda = da_2017.stack(z=("x", "y"))  # flatten each band
flattened_t_xda = flattened_xda.transpose("z", "band")
flattened_t_xda

<xarray.DataArray (z: 96615, band: 8)> Size: 2MB
dask.array<transpose, shape=(96615, 8), dtype=uint16, chunksize=(96615, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 773kB MultiIndex
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
  * x            (z) float64 773kB 3.353e+05 3.353e+05 ... 3.637e+05 3.637e+05
  * y            (z) float64 773kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
Attributes:
    crs:      epsg:EPSG:32611

In [9]:
with xr.set_options(keep_attrs=True):
    rescaled_xda = (flattened_t_xda - flattened_t_xda.mean()) / flattened_t_xda.std()
rescaled_xda

<xarray.DataArray (z: 96615, band: 8)> Size: 6MB
dask.array<truediv, shape=(96615, 8), dtype=float64, chunksize=(96615, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 773kB MultiIndex
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
  * x            (z) float64 773kB 3.353e+05 3.353e+05 ... 3.637e+05 3.637e+05
  * y            (z) float64 773kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
Attributes:
    crs:      epsg:EPSG:32611

In [10]:
print(rescaled_xda)

<xarray.DataArray (z: 96615, band: 8)> Size: 6MB
dask.array<truediv, shape=(96615, 8), dtype=float64, chunksize=(96615, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 773kB MultiIndex
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
  * x            (z) float64 773kB 3.353e+05 3.353e+05 ... 3.637e+05 3.637e+05
  * y            (z) float64 773kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
Attributes:
    crs:      epsg:EPSG:32611


In [11]:
import dask

In [12]:
client = Client(processes=False)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.103.162.49:8787/status,
Dashboard: http://10.103.162.49:8787/status,Workers: 1
Total threads: 8,Total memory: 7.70 GiB
Status: running,Using processes: False
Comm: inproc://10.103.162.49/19880/1,Workers: 0
Dashboard: http://10.103.162.49:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.103.162.49/19880/4,Total threads: 8
Dashboard: http://10.103.162.49:54381/status,Memory: 7.70 GiB
Nanny: None,


c:\Users\jcahi\OneDrive\Desktop\musa-650-spring2026\.venv\Lib\site-packages\rasterio\warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


In [13]:
X = client.persist(rescaled_xda)
X.shape

(96615, 8)

In [14]:
clf = SpectralClustering(
    n_clusters=4,
    random_state=0,
    gamma=None,
    kmeans_params={"init_max_iter": 5},
    persist_embedding=True,
)

In [15]:
from dask_ml.cluster import SpectralClustering  # NOT from sklearn

clf = SpectralClustering(
    n_clusters=4,
    random_state=0,
    gamma=None,
    kmeans_params={"init_max_iter": 5},
    persist_embedding=False,  # Turn it OFF
    n_components=100  # Uses Nyström approximation
)

%time clf.fit(X)

# Try to compute immediately in same cell
labels = clf.labels_.compute()
labels.shape

CPU times: total: 3min 11s
Wall time: 2min 5s


(96615,)

In [16]:
from dask_ml.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=0, max_iter=100)
%time kmeans.fit(X)
labels = kmeans.labels_.compute()
labels.shape

CPU times: total: 9.17 s
Wall time: 5.84 s


(96615,)

In [17]:
labels

array([3, 3, 3, ..., 0, 0, 0], shape=(96615,), dtype=int32)

In [18]:
%time clf.fit(X)

CPU times: total: 2min 14s
Wall time: 1min 4s


,n_clusters,4
,eigen_solver,None
,random_state,0
,n_init,'auto'
,gamma,None
,affinity,'rbf'
,n_neighbors,10
,eigen_tol,0.0
,assign_labels,'kmeans'
,degree,3
,coef0,1


In [19]:
labels = clf.assign_labels_.labels_.compute()
labels.shape

(96615,)

In [22]:
clf = SpectralClustering(
    n_clusters=4,
    random_state=0,
    gamma=None,
    kmeans_params={"init_max_iter": 5},
    persist_embedding=True,
)

# Use fit_predict instead of fit
%time labels = clf.fit_predict(X)

# labels is already a dask array, just compute it
labels_computed = labels.compute()
labels_computed.shape

CPU times: total: 1min 24s
Wall time: 39.3 s


FutureCancelledError: finalize-hlgfinalizecompute-c581847a8706434884ecd7d68fbfd7a9 cancelled for reason: lost dependencies.

In [ ]:
labels

In [23]:
client.close()
